# Mimakiwa Cloud Training

Trains your Mimakiwa model on Colab's free CPU runtime (NdArray backend — no GPU drivers needed).

**Steps:**
1. Run all cells top to bottom
2. Wait for training (~45–90 min on Colab CPU for 20k steps)
3. Download the 3 output files at the end
4. On your Mac: `rm -rf ~/.mimakiwa/ && mkdir ~/.mimakiwa`
5. Copy the 3 downloaded files into `~/.mimakiwa/`
6. `cargo run --release` — app loads trained model, goes straight to chat

> **Tip:** Go to Runtime → Change runtime type → GPU (T4) for ~4x faster training

In [ ]:
# ── Cell 1: Install Rust ──────────────────────────────────────────────────────
import subprocess, os

result = subprocess.run(
    'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else result.stderr[-2000:])

# Add cargo to PATH for this session
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
print('Rust version:', subprocess.check_output('rustc --version', shell=True).decode().strip())

In [ ]:
# ── Cell 2: Clone repo ────────────────────────────────────────────────────────
import subprocess

REPO_URL = 'https://github.com/aryansrao/mimakiwa.git'

r = subprocess.run(f'git clone {REPO_URL} mimakiwa', shell=True, capture_output=True, text=True)
print(r.stdout or r.stderr)

# If already cloned, just pull latest
r2 = subprocess.run('cd mimakiwa && git pull', shell=True, capture_output=True, text=True)
print(r2.stdout or r2.stderr)

In [ ]:
# ── Cell 3: Build (one-time, ~15–25 min) ─────────────────────────────────────
import subprocess, time

t0 = time.time()
print('Building mimakiwa-cloud-train (this takes 15–25 min the first time)...')

r = subprocess.run(
    'cd mimakiwa && cargo build --release -p mimakiwa-cloud-train 2>&1',
    shell=True, capture_output=True, text=True
)
print(r.stdout[-3000:] if r.stdout else r.stderr[-3000:])
print(f'Build finished in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Cell 4: Train ─────────────────────────────────────────────────────────────
import subprocess, os

os.makedirs('/content/model_out', exist_ok=True)

DATASET_URL = 'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt'
STEPS   = 20000   # increase for better quality (needs more time)
BATCH   = 8
SEQ     = 256
VOCAB   = 8192
MAX_MB  = 20

cmd = (
    f'mimakiwa/target/release/cloud-train '
    f'--dataset-url "{DATASET_URL}" '
    f'--steps {STEPS} --batch {BATCH} --seq {SEQ} '
    f'--vocab {VOCAB} --max-mb {MAX_MB} '
    f'--out /content/model_out'
)

print('Training started. Follow the loss numbers below.')
print('You can close the tab and come back — Colab keeps running for ~12 hrs.')
print('─' * 60)

# Stream output live
import sys
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('─' * 60)
print('Return code:', proc.returncode)

In [ ]:
# ── Cell 5: Download model files ──────────────────────────────────────────────
# Run this AFTER training finishes. Downloads the 3 files you need.
from google.colab import files
import os

for f in ['mimakiwa.bin', 'mimakiwa.cfg.json', 'tokenizer.json']:
    path = f'/content/model_out/{f}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1_048_576
        print(f'Downloading {f} ({size_mb:.1f} MB)...')
        files.download(path)
    else:
        print(f'MISSING: {f} — training may not have finished')

## After downloading

On your Mac, run:
```bash
rm -rf ~/.mimakiwa
mkdir ~/.mimakiwa
# Move the 3 downloaded files into ~/.mimakiwa/
mv ~/Downloads/mimakiwa.bin       ~/.mimakiwa/
mv ~/Downloads/mimakiwa.cfg.json  ~/.mimakiwa/
mv ~/Downloads/tokenizer.json     ~/.mimakiwa/

# Launch app — goes straight to Chat, no retraining
cargo run --release
```